# MySQL → OneBill Migration (Fixed + Profiled)
Bug fixes and profiling instrumentation applied. See inline comments for each change.

In [228]:
# %pip install mysql-connector-python
# %pip install sqlalchemy
# %pip install python-dotenv
import json
import pandas as pd
from sqlalchemy import create_engine
import requests
import logging
from datetime import datetime, timedelta
from concurrent.futures import ThreadPoolExecutor, as_completed
import time
import threading
import os
from collections import defaultdict

from dotenv import load_dotenv
load_dotenv(override=True)  # override=True ensures .env values take precedence


2026-05-07 10:02:50,203 [WARNING] python-dotenv could not parse statement starting at line 1
2026-05-07 10:02:50,205 [WARNING] python-dotenv could not parse statement starting at line 5
2026-05-07 10:02:50,206 [WARNING] python-dotenv could not parse statement starting at line 11


True

## Configuration

In [229]:
MAX_WORKERS = 20
TOKEN_TTL_SECONDS = 3500  # Refresh token 100s before expiry (typical OAuth TTL is 3600s)

db_url = f'mysql+mysqlconnector://{os.environ["DB_USERNAME"]}:{os.environ["DB_PASSWORD"]}@{os.environ["DB_HOST"]}/bi_custom_views'
baseUrl = 'https://sandbox-sg.onebillsoftware.com'
accessTokenURL = f'{baseUrl}/oauth/token'


## Get MySQL Data

### SQL Query

In [230]:
query = """
SELECT
    derived.`AccountCode`
    ,derived.`Current_AccountName`
    ,derived.`AccountType`
    ,derived.`OneBill_AccountType`
    ,derived.`Derived_FirstName`
    ,derived.`CreatedDate`
    ,COALESCE(grp.`TotalCount`, 1) AS `TotalCount`
    ,CASE
		WHEN grp.`TotalCount` > 1
        THEN CONCAT(derived.`Derived_FirstName`, ' ', derived.`Derived_LastName`, ' (', derived.`AccountCode`, ')')
		WHEN derived.`Derived_FirstName` = 'John' AND derived.`Derived_LastName` = 'Doe'
        THEN CONCAT(derived.`Derived_FirstName`, ' ', derived.`Derived_LastName`, ' (', derived.`AccountCode`, ')')
        ELSE derived.`Derived_LastName`
	END AS `Unique_LastName`
	,derived.`EmailAddresses`
    ,derived.`phoneHome`
    ,derived.`mobile`
    ,derived.`Address1`
    ,derived.`Address2`
    ,derived.`Suburb`
    ,derived.`City`
    ,derived.`PostCode`
    ,derived.`DateOfBirth`
FROM (
    SELECT
        `AccountCode`
        ,`AccountName` AS `Current_AccountName`
        ,`AccountType`,
        CASE
            WHEN `AccountType` = 'Actrix Residential' THEN 302
            WHEN `AccountType` = 'Residential'        THEN 303
            WHEN `AccountType` = 'HD Consumer'        THEN 305
            WHEN `AccountType` = 'Staff'              THEN 203
        END AS `OneBill_AccountType`
        ,`CreatedDate`
        ,`firstName`
        ,`lastName`
        ,CASE -- Don't want to infer first names if we already have it
			WHEN `firstName` IS NOT NULL 
            THEN `firstName`
            -- Catch any cases where there is no Account Name and set the First name to John
			WHEN `AccountName` IS NULL OR `AccountName` = ''
            THEN 'John'
            
            ELSE -- Derived first name: everything before the last space
			COALESCE(
				NULLIF(TRIM(`firstName`), ''),
				TRIM(LEFT(
					TRIM(`AccountName`),
					LENGTH(TRIM(`AccountName`)) - INSTR(REVERSE(TRIM(`AccountName`)), ' ')
				))
		)  END AS `Derived_FirstName`
        ,CASE -- Don't want to infer what the Last Name is when we already have it
			WHEN `lastName` IS NOT NULL 
            THEN `lastName`
            -- Catch any cases where there is no Account Name and set the Last name to Doe
            WHEN `AccountName` IS NULL OR `AccountName` = ''
            THEN 'Doe'
            
            ELSE COALESCE(  -- Derived last name: everything after the last space
				NULLIF(TRIM(`lastName`), ''),
				TRIM(SUBSTRING_INDEX(TRIM(`AccountName`), ' ', -1))
		)     END AS `Derived_LastName`
        ,CASE
            WHEN `EmailAddresses` IS NULL OR `EmailAddresses` = '' THEN 'someone@gmail.com'
            ELSE `EmailAddresses`
        END AS `EmailAddresses`
        ,`phoneHome`
        ,`mobile`
        ,CASE
            WHEN `addr1` IS NULL OR `addr1` = '' THEN '1 Somewhere Place'
            ELSE `addr1`
        END AS `Address1`
        ,`addr2` AS `Address2`
        ,CASE
            WHEN `suburb` IS NULL OR `suburb` = '' THEN 'N/A'
            ELSE `suburb`
        END AS `Suburb`
        ,CASE
            WHEN `city` IS NULL OR `city` = '' THEN 'Auckland'
            ELSE `city`
        END AS `City`
        ,CASE
            WHEN `postcode` IS NULL OR `postcode` = '' THEN 0000
            ELSE `postcode`
        END AS `Postcode`
        ,`dob` AS `DateOfBirth`
    FROM
        bi_custom_views.reporting_account_allColumns
    WHERE
        `AccountSegment` = 'Consumer'
) derived
LEFT JOIN (
    SELECT
        TRIM(`AccountName`)     AS `AccountName`
        ,COUNT(*)               AS `TotalCount`
        ,MIN(`AccountCode`)     AS `Min_AccountCode`
    FROM
        bi_custom_views.reporting_account_allColumns
    WHERE
        `AccountSegment` = 'Consumer'
    GROUP BY
        TRIM(`AccountName`)
    HAVING COUNT(*) > 1
) grp
    ON TRIM(derived.`Current_AccountName`) = grp.`AccountName`
ORDER BY
    derived.`Current_AccountName`,
    derived.`AccountCode`;
"""




### Turning SQL Output into a Dataframe

In [231]:
engine = create_engine(db_url)
df = pd.read_sql(query, con=engine)
df = df.head(50000)   # <-- REMOVE this line in production; it limits to 30,000 rows only
print(f'Loaded {len(df):,} rows from MySQL')

Loaded 50,000 rows from MySQL


## Log Setup

In [232]:
log_filename = f'migration_{datetime.now().strftime("%Y%m%d_%H%M%S")}.log'
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    handlers=[
        logging.FileHandler(log_filename),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)


## Token Manager (Thread-Safe)

**Bug #1 (Critical — was the main bottleneck):** `create_onebill_account()` was calling
`get_AccessToken()` on every single request, inside the thread. With 20 workers each
making hundreds of calls, this meant potentially thousands of redundant token
round-trips to the OAuth server — serialised through the GIL and hammering the
auth endpoint. A token is valid for ~1 hour; we only need to fetch it once (and
refresh it proactively before it expires).

In [233]:
class TokenManager:
    """Thread-safe bearer token cache with proactive refresh."""

    def __init__(self):
        self._lock = threading.Lock()
        self._token: str | None = None
        self._expires_at: datetime = datetime.min

    def get_token(self) -> str:
        with self._lock:           # only one thread refreshes at a time
            if datetime.now() >= self._expires_at:
                self._refresh()    # others wait at the lock, then see valid token
            return self._token

    def _refresh(self):
        logger.info('Refreshing OAuth token...')
        token_data = {
            'grant_type':    'password',
            'client_id':     os.environ['CLIENT_ID'],
            'client_secret': os.environ['CLIENT_SECRET'],
            'username':      os.environ['API_USERNAME'],
            'password':      os.environ['API_PASSWORD'],
        }
        response = requests.post(
            accessTokenURL,
            data=token_data,
            headers={'Content-Type': 'application/x-www-form-urlencoded'}
        )
        response.raise_for_status()
        payload = response.json()
        self._token = payload['access_token']
        ttl = payload.get('expires_in', TOKEN_TTL_SECONDS)
        self._expires_at = datetime.now() + timedelta(seconds=ttl - 100)  # 100s safety margin
        logger.info('Token refreshed; valid until %s', self._expires_at.strftime('%H:%M:%S'))

token_manager = TokenManager()


## Payload Builder

In [239]:
def build_account_payload(row: pd.Series) -> str:
    """Map a DataFrame row to the OneBill Account API payload."""
    row = row.where(pd.notna(row), None).to_dict()

    def serialize_date(value, fmt=None):
        if value is None:
            return None
        if hasattr(value, 'isoformat'):
            return value.strftime(fmt) if fmt else value.isoformat()
        if fmt:
            try:
                return datetime.strptime(str(value), '%Y-%m-%d').strftime(fmt)
            except ValueError:
                return str(value)
        return str(value)

    return json.dumps({
        'accountNumber':       row['AccountCode'],
        'accountName':         row['Current_AccountName'],
        'accountType':         '1001',
        'accountSubType':      row['OneBill_AccountType'],
        'activationStartDate': serialize_date(row['CreatedDate']),
        'address': [{
            'addLine1':        row['Address1'],
            'addLine2':        row['Address2'],
            'city':            row['City'],
            'state':           None,
            'country':         'New Zealand',
            'zip':             row['Postcode'],
            'defaultShipping': 'true',
            'defaultBilling':  'true',
        }],
        'contact': [{
            'firstName': row['Derived_FirstName'],
            'lastName':  row['Unique_LastName'],
            'communicationPoint': [
                {'type': 'Email', 'value': row['EmailAddresses']},
                {'type': 'Phone', 'value': row['phoneHome']},
            ],
        }],
        'accountAttribute': [{
            'key':   'Date Of Birth',
            'value': serialize_date(row['DateOfBirth'], fmt='%d/%m/%Y'),
        }],
    })


## Create Account Request

**Bug #2 (Critical):** The original `create_onebill_account` called `get_AccessToken()`
on every request, and also set `Authorization` inside the per-request headers dict —
which overrides the session-level header. We now pull the token from `TokenManager`
and let the session header do the work (updated only on refresh).

In [240]:
def create_onebill_account(session: requests.Session, base_url: str, payload: str) -> dict:
    """POST a single account to OneBill. Reuses session; token comes from TokenManager."""
    url = f'{base_url}/rest/SubscriberService/v1/subscriber'

    # Inject a fresh (possibly cached) token per-request so rotation works mid-batch.
    headers = {'Authorization': f'Bearer {token_manager.get_token()}'}

    response = session.post(url, headers=headers, data=payload, timeout=30)
    response.raise_for_status()
    data = response.json()

    validation = data.get('validationResponse', {})
    if not validation.get('successful', True):
        errors   = validation.get('validationErrorInfo', [])
        messages = '; '.join(e.get('message', '') for e in errors)
        raise ValueError(messages)

    return data


## Worker Function (with per-request timing)

In [241]:
def migrate_row(row: pd.Series, session: requests.Session) -> dict:
    account_code = row['AccountCode']
    first_name = row['Derived_FirstName']
    last_name = row['Unique_LastName']
    created_date = row['CreatedDate']

    # --- Profiling: split timing into build vs. network ---
    t0 = time.perf_counter()
    payload  = build_account_payload(row)
    t_build  = time.perf_counter() - t0

    try:
        t1       = time.perf_counter()
        response = create_onebill_account(session, baseUrl, payload)
        t_net    = time.perf_counter() - t1

        onebill_id = response.get('accountId', 'unknown')
        logger.info(f'  [OK] {account_code} — build={t_build*1000:.0f}ms  net={t_net*1000:.0f}ms')
        return {
            'account_code': account_code,
            'first_name': first_name,
            'last_name': last_name,
            'status':       'success',
            'error':        None,
            'elapsed_build_ms': round(t_build * 1000, 1),
            'elapsed_net_ms':   round(t_net   * 1000, 1),
        }

    except Exception as e:
        t_net = time.perf_counter() - t1 if 't1' in dir() else 0
        logger.error(f'  [FAIL] {account_code} — {e}')
        return {
            'account_code': account_code,
            'first_name': first_name,
            'last_name': last_name,
            'status':       'failed',
            'error':        str(e),
            'elapsed_build_ms': round(t_build * 1000, 1),
            'elapsed_net_ms':   round(t_net   * 1000, 1),
        }


## Migrate Function

**Bug #3 (NameError crash):** In the original code the `futures` dict was built
*before* `session`, `rows`, and `total` were defined. Python executes the dict
comprehension immediately, so it raises `NameError: name 'rows' is not defined`
before any work happens. The variables must be defined first.

**Bug #4 (signature mismatch):** `migrate_row(row, session)` takes 2 arguments but
the original `migrate` passed 3 — `(row, session, token_holder)`. This would raise
`TypeError` at runtime on every submitted future.

**Bug #5 (duplicate `futures` assignment):** The original built `futures` twice —
once before the session existed (crash) and once inside the `with` block. Only the
second one was intended; the first is dead/broken code.

**Bug #6 (timeout too tight):** The original timeout was 10 seconds. If OneBill
takes >10 s under load (not unusual for billing APIs) every request times out.
Increased to 30 s.

**Performance note:** Python's GIL means CPU-bound work doesn't parallelise with
`ThreadPoolExecutor`. For this workload the bottleneck is *network latency*, so
threads help — but only up to the point where the server is the constraint. Check
the profiling output to see whether `elapsed_net_ms` dominates; if so, the server
is rate-limiting or saturated and more workers won't help.

In [242]:
def migrate(df: pd.DataFrame, max_workers: int = MAX_WORKERS) -> pd.DataFrame:
    """Migrate every row in df to OneBill, returning a results DataFrame."""

    # --- FIX #3/#5: define all variables BEFORE building the futures dict ---
    session = requests.Session()
    adapter = requests.adapters.HTTPAdapter(
        pool_connections=max_workers,
        pool_maxsize=max_workers
    )
    session.mount('https://', adapter)
    session.headers.update({
        'proxy_accountNumber': '31802',   # Replace with your partner account number
        'Content-Type': 'application/json',
    })

    rows  = [row for _, row in df.iterrows()]
    total = len(rows)
    results: list[dict] = []

    logger.info(f'Starting migration of {total:,} records with {max_workers} workers...')
    wall_start = time.perf_counter()

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        # --- FIX #4: only 2 args — token now comes from TokenManager inside migrate_row ---
        futures = {
            executor.submit(migrate_row, row, session): row['AccountCode']
            for row in rows
        }

        for i, future in enumerate(as_completed(futures), start=1):
            result = future.result()
            results.append(result)

            if i % 50 == 0 or i == total:
                ok   = sum(1 for r in results if r['status'] == 'success')
                fail = sum(1 for r in results if r['status'] == 'failed')
                logger.info(f'Progress: {i}/{total} — {ok} ok, {fail} failed')

    wall_elapsed = time.perf_counter() - wall_start

    results_df = pd.DataFrame(results)
    success = (results_df['status'] == 'success').sum()
    failed  = (results_df['status'] == 'failed').sum()

    logger.info(
        f'Migration done in {wall_elapsed:.1f}s — '
        f'{success} succeeded, {failed} failed. (log: {log_filename})'
    )

    # --- Profiling summary ---
    print('\n=== Profiling Summary ===')
    print(f'Total wall time:          {wall_elapsed:.1f}s')
    print(f'Throughput:               {total / wall_elapsed:.1f} accounts/s')
    print(f'Avg build time per row:   {results_df["elapsed_build_ms"].mean():.1f}ms')
    print(f'Avg network time per row: {results_df["elapsed_net_ms"].mean():.1f}ms')
    print(f'Max network time:         {results_df["elapsed_net_ms"].max():.1f}ms')
    print(f'P95 network time:         {results_df["elapsed_net_ms"].quantile(0.95):.1f}ms')
    print('=========================')

    return results_df


## Run Migration

In [243]:
results_df = migrate(df)

failures = results_df[results_df['status'] == 'failed']
print(f'\nFailed rows ({len(failures)}):')
display(failures)


2026-05-07 10:04:29,917 [INFO] Starting migration of 50,000 records with 20 workers...
2026-05-07 10:04:29,921 [INFO] Refreshing OAuth token...
2026-05-07 10:04:31,497 [INFO] Token refreshed; valid until 11:02:50
2026-05-07 10:05:04,377 [INFO]   [OK] 99967328 — build=0ms  net=34421ms
2026-05-07 10:05:04,392 [INFO]   [OK] 99965756 — build=1ms  net=34439ms
2026-05-07 10:05:04,463 [INFO]   [OK] platypus-64422 — build=1ms  net=34531ms
2026-05-07 10:05:04,464 [INFO]   [OK] platypus-58106 — build=1ms  net=34534ms
2026-05-07 10:05:04,490 [INFO]   [OK] platypus-83552 — build=1ms  net=34566ms
2026-05-07 10:05:04,514 [INFO]   [OK] platypus-76977 — build=0ms  net=34564ms
2026-05-07 10:05:04,576 [INFO]   [OK] 94041444 — build=1ms  net=34656ms
2026-05-07 10:05:04,592 [INFO]   [OK] platypus-56638 — build=1ms  net=34657ms
2026-05-07 10:05:04,621 [INFO]   [OK] 99967687 — build=1ms  net=34673ms
2026-05-07 10:05:04,666 [INFO]   [OK] 99968511 — build=0ms  net=34710ms
2026-05-07 10:05:05,276 [INFO]   [OK]


=== Profiling Summary ===
Total wall time:          8729.1s
Throughput:               5.7 accounts/s
Avg build time per row:   1.2ms
Avg network time per row: 3486.7ms
Max network time:         94210.6ms
P95 network time:         5060.2ms

Failed rows (4288):


,account_code,first_name,last_name,status,error,elapsed_build_ms,elapsed_net_ms
20,99999883,Accounts,Payable,failed,Accounting Name [Accounts Payable] already exist.,0.5,1836.8
30,99992910,Rosemarie Gail,Cowling,failed,Account number 99992910 already exists.,0.5,269.6
31,99961175,Robert,Storey,failed,Account number 99961175 already exists.,0.3,324.9
32,99969251,Steve,Mantle,failed,Account number 99969251 already exists.,0.6,368.6
64,99993984,Accounts,Payable,failed,Accounting Name [Accounts Payable] already exist.,0.5,1810.1
...,...,...,...,...,...,...,...
49888,99986731,Justis,Glastonbury,failed,Account number 99986731 already exists.,0.8,270.1
49899,99974158,Justyn,Armstrong,failed,Accounting Name [Justyn Armstrong] already ex...,0.8,2103.6
49914,94028165,K & E,Buswell,failed,Account number 94028165 already exists.,2.2,269.9
49916,94030459,K & D,Trye,failed,Account number 94030459 already exists.,0.9,328.7


In [ ]:
failures.to_csv(f'Failed_Migrations_{datetime.now().strftime("%Y%m%d_%H%M%S")}.csv', index=False)
